# Meta-Model 4 — Week 1-2 Baseline Check

This notebook trains and evaluates **FINAL META-MODEL 4** from `mental_health_final_model.py` using the production pipeline in `src/`.

**Architecture**
1. `processing.py` — clean + engineer features + `High_Risk = PHQ_Total ≥ 15`
2. SMOTETomek resampling with PHQ-aware sample weights (PHQ==15 → 0.40)
3. Calibrated base models: RF (isotonic) + XGB (sigmoid) + CatBoost (sigmoid)
4. Meta-learner: XGBoost on stacked base probabilities (`final_xgb`)

Artifacts are written to `../artifacts/` for the FastAPI service.

In [ ]:
from pathlib import Path
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.processing import (
    engineer_features_and_target,
    load_and_clean_raw_data,
    prepare_for_modeling,
)
from src.models.meta_model4 import (
    MetaModel4Config,
    evaluate_binary_model,
    train_meta_model4,
)

print("Project root:", PROJECT_ROOT)

## 1. Configure data path

Point `DATA_PATH` at your TSC survey Excel export (same file used in the Colab notebook).

In [ ]:
# --- EDIT THIS PATH ---
DATA_PATH = Path.home() / "Downloads" / (
    "TEACHERS SERVICE COMMISSION SURVEY ON THE FEATURES "
    "CONTRIBUTING TO TEACHER DEPRESSION (1-1933).xlsx"
)

# Fallback: any xlsx under Downloads matching TEACHERS / SURVEY
if not DATA_PATH.exists():
    candidates = list(Path.home().joinpath("Downloads").glob("*TEACHERS*SURVEY*.xlsx"))
    candidates += list(Path.home().joinpath("Downloads").glob("*TEACHER*.xlsx"))
    if candidates:
        DATA_PATH = candidates[0]

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)
RANDOM_STATE = 42
TEST_SIZE = 0.2

print("Data path:", DATA_PATH)
print("Exists:", DATA_PATH.exists())
assert DATA_PATH.exists(), (
    "Survey Excel not found. Set DATA_PATH to your TSC survey file."
)

## 2. Load & engineer features (aligned with `processing.py`)

In [ ]:
df_raw = load_and_clean_raw_data(str(DATA_PATH))
df = engineer_features_and_target(df_raw)
X, y, feature_order = prepare_for_modeling(df)
phq = df.loc[X.index, "PHQ_Total"]

print("Engineered shape:", df.shape)
print("X shape:", X.shape)
print("n_features:", len(feature_order))
print("High_Risk rate:", f"{y.mean():.1%}")
print("PHQ_Total describe:\n", phq.describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.countplot(x=y, ax=axes[0])
axes[0].set_title("High_Risk distribution")
axes[0].set_xlabel("High Risk (0=Low, 1=High)")
sns.histplot(phq, bins=28, kde=True, ax=axes[1], color="steelblue")
axes[1].axvline(15, color="red", ls="--", label="High_Risk cutoff (≥15)")
axes[1].set_title("PHQ_Total")
axes[1].legend()
plt.tight_layout()
plt.show()

## 3. Train / test split (stratified)

In [ ]:
X_train, X_test, y_train, y_test, phq_train, phq_test = train_test_split(
    X, y, phq,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train High_Risk:", f"{y_train.mean():.1%}", "| Test:", f"{y_test.mean():.1%}")

## 4. Train Meta-Model 4

Same hyperparameters as the notebook **FINAL META-MODEL 4** block.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)
log = logging.getLogger("meta4")

cfg = MetaModel4Config(random_state=RANDOM_STATE, test_size=TEST_SIZE)
model, train_info = train_meta_model4(
    X_train=X_train,
    y_train=y_train,
    phq_train=phq_train,
    feature_order=feature_order,
    cfg=cfg,
    logger=log,
)
train_info

## 5. Hold-out evaluation

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("=== META-MODEL 4 (Final) ===")
print(classification_report(y_test, y_pred, target_names=["Low Risk", "High Risk"]))

metrics = evaluate_binary_model(model, X_test, y_test)
print(
    f"Macro F1={metrics['macro_f1']:.4f} | "
    f"High-Risk F1={metrics['f1_high_risk']:.4f} | "
    f"ROC-AUC={metrics['roc_auc']:.4f} | "
    f"AP={metrics['average_precision']:.4f}"
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
    xticklabels=["Pred Low", "Pred High"],
    yticklabels=["Actual Low", "Actual High"],
)
axes[0].set_title("Confusion Matrix — Meta-Model 4")

RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1], name="Meta-Model 4")
axes[1].plot([0, 1], [0, 1], "--", color="gray")
axes[1].set_title("ROC Curve")

PrecisionRecallDisplay.from_predictions(y_test, y_proba, ax=axes[2], name="Meta-Model 4")
axes[2].set_title("Precision–Recall")

plt.tight_layout()
plt.show()

## 6. Base-model contribution (meta-features on test)

In [ ]:
meta_test = model.meta_features(X_test)
meta_df = pd.DataFrame(meta_test, columns=list(model.META_FEATURE_NAMES))
meta_df["y_true"] = y_test.to_numpy()
meta_df["y_pred"] = y_pred
meta_df["y_proba"] = y_proba

display(meta_df.describe())

fig, ax = plt.subplots(figsize=(7, 4))
meta_df[list(model.META_FEATURE_NAMES)].plot.box(ax=ax)
ax.set_title("Calibrated base P(High Risk) on test set")
ax.set_ylabel("Probability")
plt.tight_layout()
plt.show()

## 7. Save artifacts for FastAPI

Same files produced by:
```bash
python src/models/train.py --data_path <survey.xlsx> --model meta4
```

In [ ]:
joblib.dump(model, ARTIFACTS_DIR / "meta_model4.joblib")
joblib.dump(model.final_xgb, ARTIFACTS_DIR / "final_xgb.joblib")
joblib.dump(model.cal_rf, ARTIFACTS_DIR / "cal_rf.joblib")
joblib.dump(model.cal_xgb, ARTIFACTS_DIR / "cal_xgb.joblib")
joblib.dump(model.cal_cat, ARTIFACTS_DIR / "cal_cat.joblib")
joblib.dump(feature_order, ARTIFACTS_DIR / "feature_order.joblib")

metadata = {
    "model_type": "MetaModel4",
    "source": "notebooks/meta_model4_baseline.ipynb",
    "n_features": len(feature_order),
    "metrics_test": {
        "macro_f1": round(metrics["macro_f1"], 4),
        "f1_high_risk": round(metrics["f1_high_risk"], 4),
        "roc_auc": round(metrics["roc_auc"], 4),
        "average_precision": round(metrics["average_precision"], 4),
    },
    "train_info": train_info,
}
joblib.dump(metadata, ARTIFACTS_DIR / "model_metadata.joblib")

print("Saved to", ARTIFACTS_DIR.resolve())
for p in sorted(ARTIFACTS_DIR.glob("*.joblib")):
    print(" -", p.name, f"({p.stat().st_size // 1024} KB)")

## 8. Smoke-test reload (API path)

Confirms `meta_model4.joblib` round-trips for serving.

In [ ]:
loaded = joblib.load(ARTIFACTS_DIR / "meta_model4.joblib")
sample = X_test.iloc[:3]
print("predict:", loaded.predict(sample))
print("predict_proba:\n", loaded.predict_proba(sample))
print("OK — Meta-Model 4 ready for uvicorn src.api.main:app")